In [ ]:
# ==============================================================================
# PARALLEL FACT: RENTAL
# ==============================================================================
from notebooks.helpers import IncrementalPipeline, TableConfig, get_latest_batch_id, setup_logger, safe_count, generate_batch_id
import pandas as pd
from notebooks.helpers.silver_transforms import transform_rental_fact

logger = setup_logger("parallel_fact_rental")

batch_id = generate_batch_id()
pipeline = IncrementalPipeline(spark, dbutils, batch_id=batch_id)

dependencies = ["staff", "inventory", "payment"]

bronze_batch_id = get_latest_batch_id(spark, "rental")
if not bronze_batch_id:
    raise ValueError("No bronze batch_id found for rental; run bronze load first.")
logger.info(f"Using bronze batch_id for rental: {bronze_batch_id}")

config = TableConfig(
    table_name="rental",
    business_key="rental_id",
    surrogate_key="rental_key",
    watermark_column="rental_date",
    scd_type=1,
    gold_table_name="fact_rental",
    silver_transform=transform_rental_fact,
    dependencies=dependencies,
)

print("Row Counts (Before):")
print(f"fact_rental: {safe_count(spark, 'fact_rental')}")
print("\nLatest Watermarks (Before):")
display(spark.table("wheelie.monitoring.watermarks"))

results = pipeline.load_tables([config], force_full=False, bronze_batch_id=bronze_batch_id)
display(pd.DataFrame(results))

print("\nRow Counts (After):")
print(f"fact_rental: {safe_count(spark, 'fact_rental')}")
print("\nLatest Watermarks (After):")
display(spark.table("wheelie.monitoring.watermarks"))
